# Assignment 2 — Hyperparameter Tuning

**Purpose:** Tune the validation-selected architecture with repeated-seed rolling random search.

This notebook is an executable evidence view over the shared local pipeline. The
command-line runner and notebook call the same stage implementation, preventing code
drift. Completed artifacts are reused; no Google Colab or Google Drive runtime is used.


In [1]:
# Local WSL project bootstrap. Dependency installation belongs to the
# locked project environment and is intentionally not repeated in notebooks.
from pathlib import Path
import json
import os
import platform
import sys

candidate = Path.cwd().resolve()
PROJECT_ROOT = next(
    (path for path in [candidate, *candidate.parents] if (path / "shared").exists()),
    candidate,
)
if not (PROJECT_ROOT / "shared").exists():
    raise FileNotFoundError(
        "Open this notebook from the local odl-assignment project root."
    )
os.environ["ODL_PROJECT_ROOT"] = str(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from shared.config import PATHS
from shared.runtime import RuntimeOptions
from shared.workflow import run_stage

OPTIONS = RuntimeOptions.from_environment()
print("Project root:", PROJECT_ROOT)
print("Run mode:", OPTIONS.mode)
print("Device policy:", OPTIONS.device)
print("WSL distribution:", os.getenv("WSL_DISTRO_NAME", "not detected"))


I0000 00:00:1786540927.503427    5650 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1786540927.556862    5650 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Project root: /home/nicholas/code/odl-assignment
Run mode: smoke
Device policy: gpu
WSL distribution: Ubuntu-24.04


I0000 00:00:1786540928.777410    5650 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [2]:
# Run the shared stage or load its verified completed result.
STAGE_SUMMARY = run_stage("tune", options=OPTIONS)
STAGE_SUMMARY


{'architecture': 'lstm',
 'candidate_count': 2,
 'validation_fits': 2,
 'selected_candidate': 'candidate_08',
 'formulation': 'residual',
 'validation_mae': 11.208906444409253,
 'epoch_budget': 1,
 'training_seconds': 4.614089546999821,
 'run_mode': 'smoke'}

In [3]:
# Record the exact runtime used for this output when preflight evidence exists.
environment_path = PATHS.environment_artifacts / f"environment_{OPTIONS.mode}.json"
ENVIRONMENT = (
    json.loads(environment_path.read_text(encoding="utf-8"))
    if environment_path.exists()
    else {"status": "preflight evidence not yet generated"}
)
ENVIRONMENT


{'captured_at_utc': '2026-08-12T13:18:29.136112+00:00',
 'project_root': '/home/nicholas/code/odl-assignment',
 'platform': 'Linux-6.6.87.2-microsoft-standard-WSL2-x86_64-with-glibc2.39',
 'system': 'Linux',
 'release': '6.6.87.2-microsoft-standard-WSL2',
 'python': '3.12.3',
 'wsl_distribution': 'Ubuntu-24.04',
 'device': 'gpu',
 'tensorflow_gpu_devices': ['/physical_device:GPU:0'],
 'batch_size': 256,
 'mixed_precision': False,
 'nvidia_smi': {'available': True,
  'name': 'NVIDIA GeForce RTX 3050 Laptop GPU',
  'driver_version': '591.74',
  'memory.total': '4096',
  'memory.used': '665',
  'utilization.gpu': '3',
  'temperature.gpu': '57',
  'power.draw': '18.22'},
 'packages': {'tensorflow': '2.21.0',
  'numpy': '2.3.5',
  'pandas': '2.3.3',
  'pyarrow': '25.0.0',
  'scikit-learn': '1.9.0',
  'matplotlib': '3.11.1',
  'seaborn': '0.13.2'},
 'git': {'available': True,
  'revision': '8cbcae3e1948b87b42012417b61d73ab859f470e',
  'dirty': True},
 'experiment_config': {'methodology_versi

## Stratified random-search evidence

**Decision.** Evaluate 16 deterministic candidates across three rolling folds and seeds
42/52. Two anchors preserve the basic and previous-best settings; remaining candidates
balance direct and persistence-residual formulations. Random search is efficient when
only some dimensions strongly affect performance
([Bergstra and Bengio](https://www.jmlr.org/papers/v13/bergstra12a.html)). **Alternatives.**
The previous single-execution search result was smaller than seed variation.
**Trade-off.** Six validation fits per candidate cost more but make selection more
defensible. The frozen test benchmark is never read during selection.


In [4]:
import pandas as pd

suffix = OPTIONS.artifact_suffix
trials = pd.read_csv(
    PATHS.assignment2_tables / f"random_search_rolling_results{suffix}.csv"
)
summary = pd.read_csv(
    PATHS.assignment2_tables / f"random_search_summary{suffix}.csv"
)
best_hyperparameters = json.loads(
    (PATHS.models / f"best_hyperparameters{suffix}.json").read_text(encoding="utf-8")
)
display(summary)
display(trials)
display(best_hyperparameters)


,candidate_id,mean_mae,std_mae,folds,mean_seed_std_mae,min_seeds_per_fold,max_seeds_per_fold,evaluations,mean_best_epoch,median_best_epoch,parameters,num_layers,formulation,se_mae
0,candidate_08,11.208906,NaN,1,0.0,1,1,1,1.0,1.0,95361,1,residual,0.0
1,candidate_00,41.389352,NaN,1,0.0,1,1,1,1.0,1.0,29249,1,direct,0.0


,candidate_id,formulation,units,dense_units,dropout,learning_rate,l2_strength,num_layers,huber_delta,source,...,source_fingerprint,n,mae,rmse,r2,smape_percent,best_epoch,epochs_recorded,parameters,training_seconds
0,candidate_00,direct,64,32,0.2,0.001000,0.0,1,10.0,basic_anchor,...,5bd159fcfc91988987f735c6bcbdd93d340a9bde97152f...,8469,41.389352,65.212473,0.127332,61.181542,1,1,29249,3.381348
1,candidate_08,residual,128,64,0.3,0.000636,0.0,1,5.0,random,...,5bd159fcfc91988987f735c6bcbdd93d340a9bde97152f...,8469,11.208906,22.367085,0.897339,21.192620,1,1,95361,3.473740


{'architecture': 'lstm',
 'objective': 'mean_rolling_validation_mae',
 'values': {'candidate_id': 'candidate_08',
  'formulation': 'residual',
  'units': 128,
  'dense_units': 64,
  'dropout': 0.3,
  'learning_rate': 0.0006358361806606889,
  'l2_strength': 0.0,
  'num_layers': 1,
  'huber_delta': 5.0,
  'source': 'random'},
 'epoch_budget': 1,
 'selection_rule': 'one-standard-error; parameters, layers, direct formulation',
 'run_mode': 'smoke',
 'config_hash': '3c74d277bc97ef6a',
 'source_fingerprint': '5bd159fcfc91988987f735c6bcbdd93d340a9bde97152f2b8fdc12c7edd821a1',
 'selection_audit': {'best_candidate_by_mean_mae': 'candidate_08',
  'best_mean_mae': 11.208906444409253,
  'best_fold_level_standard_error': 0.0,
  'one_standard_error_threshold': 11.208906444409253,
  'eligible_candidates': ['candidate_08'],
  'selected_candidate': 'candidate_08',
  'tie_break_reason': 'Among candidates within one fold-level standard error, prefer fewer parameters, then fewer LSTM layers, then direct f

## Final validation selection and fixed official model

**Decision.** Average seeds within each fold, estimate standard error across the three
future folds, then apply the one-standard-error rule, preferring fewer parameters, fewer
layers, then a direct formulation when performance is practically tied. The selected
configuration is trained once with seed 42 on all pre-test data for the median
validation-derived epoch budget. Huber loss limits the influence of extreme spikes
([Huber](https://doi.org/10.1214/aoms/1177703732)). **Trade-off.** Simplicity may give up
a tiny mean-MAE advantage in exchange for stability and clearer attribution.


In [5]:
final_selection = json.loads(
    (PATHS.models / f"final_model_selection{suffix}.json").read_text(encoding="utf-8")
)
display(final_selection)
print("Rolling tuning and single-model selection complete.")


{'architecture': 'lstm',
 'selected_candidate': {'candidate_id': 'candidate_08',
  'formulation': 'residual',
  'units': 128,
  'dense_units': 64,
  'dropout': 0.3,
  'learning_rate': 0.0006358361806606889,
  'l2_strength': 0.0,
  'num_layers': 1,
  'huber_delta': 5.0,
  'source': 'random'},
 'selected_seed': 42,
 'epoch_budget': 1,
 'selection_metric': 'mean_rolling_validation_mae',
 'mean_validation_mae': 11.208906444409253,
 'validation_standard_error': 0.0,
 'final_checkpoint': '/home/nicholas/code/odl-assignment/artifacts/models/tuned_final_smoke.keras',
 'frozen_test_used_for_selection': False,
 'selection_locked_before_benchmark': True,
 'run_mode': 'smoke',
 'config_hash': '3c74d277bc97ef6a',
 'source_fingerprint': '5bd159fcfc91988987f735c6bcbdd93d340a9bde97152f2b8fdc12c7edd821a1',
 'selection_audit': {'best_candidate_by_mean_mae': 'candidate_08',
  'best_mean_mae': 11.208906444409253,
  'best_fold_level_standard_error': 0.0,
  'one_standard_error_threshold': 11.208906444409253

Rolling tuning and single-model selection complete.
